# 中金價量因子批量測試

沿用 `single_factor_test.ipynb` 的資料準備流程，計算 `zhong_jin_vol_price.py` 的全部公開因子，並以多進程執行單因子回測。

最終只保留同時符合以下條件的因子：
- `|Rank_IC| > 0.02`
- `|Rank_IC_IR| > 0.5`
- `|t統計量| > 2`

In [1]:
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import warnings

# 因子計算交由多進程平行；限制每個 Polars 子進程的內部執行緒，避免過度競爭
os.environ.setdefault("POLARS_MAX_THREADS", "1")
warnings.filterwarnings("ignore", category=FutureWarning)

# 從專案內任意常見位置啟動 notebook 都能找到 src
_cwd = Path.cwd().resolve()
root = next(
    (p for p in (_cwd, *_cwd.parents) if (p / "src").is_dir()),
    Path("/home/newt/quant_project"),
)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

print(f"Project root: {root}")

Project root: /home/newt/quant_project


In [2]:
import polars as pl
from src.factor_engine import FactorEngine

RAM_PATH = "/dev/shm/qlib_ram"
start_time = "2015-06-01"
end_time = "2026-05-29"

# 參考 factor_test copy.ipynb 建立 FactorEngine；此處只取 open 的完整索引
engine = FactorEngine(
    qlib_data_path=RAM_PATH,
    instrument="all",
    start_time=start_time,
    end_time=end_time,
    device="cuda:0",
    max_backtrack_days=0,
    max_future_days=0,
    features=["open"],
    adjust_factor="none",
)

engine_index = engine.calc_df("$open", col_name="_open").index.to_frame(index=False)
engine_index.columns = ["date", "symbol"]
engine_pairs = (
    pl.from_pandas(engine_index)
    .with_columns(
        pl.col("date").cast(pl.Date),
        pl.col("symbol").cast(pl.String),
    )
    .sort(["symbol", "date"])
)

del engine_index
print(f"FactorEngine 基準：{engine_pairs.height:,} 個 date/symbol pairs")
engine_pairs.head()

data path: /dev/shm/qlib_ram


[1272428:MainThread](2026-07-31 18:40:16,401) INFO - qlib.Initialization - [config.py:452] - default_conf: client.
[1272428:MainThread](2026-07-31 18:40:16,585) INFO - qlib.Initialization - [__init__.py:82] - qlib successfully initialized based on client settings.
[1272428:MainThread](2026-07-31 18:40:16,586) INFO - qlib.Initialization - [__init__.py:84] - data_path={'__DEFAULT_FREQ': PosixPath('/dev/shm/qlib_ram')}


[FactorEngine] Loading data: instrument=all, period=[2015-06-01, 2026-05-29], device=cuda:0
[StockData] 載入特徵 (1): ['open']
[FactorEngine] Data loaded: 2682 days, 1944 stocks, 1 features, tensor shape=(2682, 1, 1944)
FactorEngine 基準：5,213,808 個 date/symbol pairs


date,symbol
date,str
2015-06-01,"""1101"""
2015-06-02,"""1101"""
2015-06-03,"""1101"""
2015-06-04,"""1101"""
2015-06-05,"""1101"""


In [3]:
import duckdb

# 讀取 DuckDB，再以 FactorEngine 的完整 date/symbol 面板為準對齊
duckdb_path = "/home/newt/quant_project/duckdb/daily_data.duckdb"
con = duckdb.connect(database=duckdb_path, read_only=True)
duck_df = (
    con.execute("SELECT * FROM daily_data").pl()
    .with_columns(
        pl.col("date").cast(pl.Date),
        pl.col("symbol").cast(pl.String),
        pl.lit(True).alias("_duck_present"),
    )
)

# Qlib/Engine 是主索引；基準外的 DuckDB 資料只報告、不併入
outside_engine = duck_df.select(["date", "symbol"]).join(
    engine_pairs, on=["date", "symbol"], how="anti"
)
if outside_engine.height:
    print(f"注意：DuckDB 有 {outside_engine.height:,} 個 pairs 不在 Qlib/Engine 基準內，將不納入")

df = (
    engine_pairs.join(duck_df, on=["date", "symbol"], how="left")
    .sort(["symbol", "date"])
)
missing_pair_count = df["_duck_present"].null_count()

# OHLC 沿用同一股票前一個交易日；其餘未指定欄位保留 null
price_cols = ["open", "high", "low", "close"]
df = (
    df.with_columns(
        [pl.col(col).forward_fill().over("symbol").alias(col) for col in price_cols]
    )
    .with_columns(
        pl.col("volume").fill_null(0),
        pl.col("return_day").fill_null(0),
    )
    .drop("_duck_present")
)

con.close()
print(f"DuckDB 原始：{duck_df.height:,} rows")
print(f"Engine 對齊後：{df.height:,} rows（新增並填充 {missing_pair_count:,} 個 pairs）")
df.select(["date", "symbol", *price_cols, "volume", "return_day"]).head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

DuckDB 原始：4,708,990 rows
Engine 對齊後：5,213,808 rows（新增並填充 504,818 個 pairs）


date,symbol,open,high,low,close,volume,return_day
date,str,f64,f64,f64,f64,f64,f64
2015-06-01,"""1101""",41.7,42.1,41.05,41.1,6324.0,-1.4388
2015-06-02,"""1101""",41.1,41.1,40.3,40.4,8592.0,-1.7032
2015-06-03,"""1101""",40.4,40.55,40.0,40.15,6614.0,-0.6188
2015-06-04,"""1101""",40.0,40.0,37.65,38.8,13609.0,-3.3624
2015-06-05,"""1101""",38.75,39.1,38.55,38.95,8041.0,0.3866


In [4]:
price_cols = ['open', 'high', 'low', 'close']
adj_col = 'adj_factor'

# 原始復權因子必須為正；engine 補出的日期其 adj_factor 為 null，計算時視為 1
invalid_adj_count = df.filter(
    pl.col(adj_col).is_not_null() & (pl.col(adj_col) <= 0)
).height
if invalid_adj_count:
    raise ValueError(f"adj_factor 有 {invalid_adj_count:,} 筆小於或等於 0，復權無法進行")

# 每檔股票按日期由舊到新累乘當日復權因子
df = (
    df.sort(['symbol', 'date'])
    .with_columns(
        pl.col(adj_col)
        .fill_null(1.0)
        .cum_prod()
        .over('symbol')
        .alias('cum_adj_factor')
    )
    .with_columns(
        [
            (pl.col(col) * pl.col('cum_adj_factor')).alias(f'{col}_adj')
            for col in price_cols
        ]
    )
)

df.select(['symbol', 'date', adj_col, 'cum_adj_factor', *[f'{col}_adj' for col in price_cols]]).head()

symbol,date,adj_factor,cum_adj_factor,open_adj,high_adj,low_adj,close_adj
str,date,f64,f64,f64,f64,f64,f64
"""1101""",2015-06-01,1.0,1.0,41.7,42.1,41.05,41.1
"""1101""",2015-06-02,1.0,1.0,41.1,41.1,40.3,40.4
"""1101""",2015-06-03,1.0,1.0,40.4,40.55,40.0,40.15
"""1101""",2015-06-04,1.0,1.0,40.0,40.0,37.65,38.8
"""1101""",2015-06-05,1.0,1.0,38.75,39.1,38.55,38.95


In [5]:
from src.factor.single_factor_eval import prepare_eval_returns

# 通用未來報酬 / lag 報酬寫進原始 df，驗證時直接用
df = prepare_eval_returns(df, price_col="close_adj")
df.select(
    ["symbol", "date", "close_adj", "1day_return", "3day_return", "returns_lag0", "returns_lag1"]
).head()

2026-07-31 18:40:42 - panda_common.logger_config - INFO - 从配置文件加载配置: /home/newt/quant_project/panda_factor/panda_common/panda_common/config.yaml
2026-07-31 18:40:42 - panda_common.logger_config - INFO - 从环境变量更新配置: USER
2026-07-31 18:40:42 - panda_common.logger_config - INFO - 初始化配置成功: {'HUB_START_DATE': 20170101, 'HUB_END_DATE': 20250321, 'UPDATE_TIME': '20:00', 'STOCKS_UPDATE_TIME': '20:00', 'FACTOR_UPDATE_TIME': '20:30', 'DATASOURCE': 'tqsdk', 'DATAHUBSOURCE': '', 'MUSER': '', 'MPASSWORD': '', 'TS_TOKEN': '', 'XT_TOKEN': '', 'USER': 'newt', 'PASSWORD': 'xxxxxx', 'MONGO_USER': 'panda', 'MONGO_PASSWORD': 'panda', 'MONGO_URI': '127.0.0.1:27017', 'MONGO_AUTH_DB': 'admin', 'MONGO_DB': 'panda', 'MONGO_TYPE': 'replica_set', 'MONGO_REPLICA_SET': 'rs0', 'LLM_API_KEY': '这里填写你的KEY', 'LLM_MODEL': 'deepseek-chat', 'LLM_BASE_URL': 'https://api.deepseek.com/v1', 'LOG_LEVEL': 'DEBUG', 'log_file': 'logs/data_cleaner.log', 'log_rotation': '1 MB'}


symbol,date,close_adj,1day_return,3day_return,returns_lag0,returns_lag1
str,date,f64,f64,f64,f64,f64
"""1101""",2015-06-01,41.1,-0.006188,-0.035891,-0.006188,-0.033624
"""1101""",2015-06-02,40.4,-0.033624,-0.006227,-0.033624,0.003866
"""1101""",2015-06-03,40.15,0.003866,0.001289,0.003866,0.02439
"""1101""",2015-06-04,38.8,0.02439,-0.003851,0.02439,-0.026316
"""1101""",2015-06-05,38.95,-0.026316,-0.04386,-0.026316,-0.001287


In [6]:
import gc

from src.alphas import zhong_jin_vol_price as zj

# 需傳 days 的因子使用模組定義的三個標準週期；固定因子使用函式預設週期
PERIODS = (zj.DAYS_1M, zj.DAYS_3M, zj.DAYS_6M)
factor_tasks = [
    (factor_name, days)
    for factor_name in zj.PERIOD_FACTORS
    for days in PERIODS
] + [(factor_name, None) for factor_name in zj.FIXED_FACTORS]

# 不再使用的物件先釋放
for _name in ("engine", "engine_pairs", "duck_df", "outside_engine"):
    globals().pop(_name, None)
gc.collect()

assert len(factor_tasks) == len(zj.PERIOD_FACTORS) * len(PERIODS) + len(zj.FIXED_FACTORS)
print(f"待測試：{len(factor_tasks)} 組因子（單進程迴圈）")


待測試：95 組因子（單進程迴圈）


In [7]:
import io
from contextlib import redirect_stdout

import src.factor.single_factor_eval as single_factor_eval


def _metric_as_float(value):
    """panda_factor 的統計值可能是字串，統一轉成 float。"""
    return float(str(value).replace("%", "").strip())


def _run_factor(task):
    """計算一個因子並呼叫 run_single_factor_test。"""
    base_name, days = task
    task_name = f"{base_name}_{days}d" if days is not None else base_name

    try:
        generator = getattr(zj, f"generate_{base_name}")
        before = set(df.columns)
        factor_df = generator(df, days) if days is not None else generator(df)
        new_columns = [column for column in factor_df.columns if column not in before]
        if len(new_columns) != 1:
            raise RuntimeError(f"預期新增 1 欄，實際新增 {new_columns}")
        factor_name = new_columns[0]

        original_display = single_factor_eval.display
        single_factor_eval.display = lambda _: None
        try:
            with redirect_stdout(io.StringIO()):
                alpha = single_factor_eval.run_single_factor_test(
                    factor_df,
                    factor_name=factor_name,
                    group_count=5,
                    show_plots=False,
                )
        finally:
            single_factor_eval.display = original_display

        rank_ic = _metric_as_float(alpha.df_info2.at["Rank_IC", factor_name])
        rank_ic_ir = _metric_as_float(alpha.df_info2.at["Rank_IC_IR", factor_name])
        t_stat = _metric_as_float(alpha.df_info2.at["t統計量", factor_name])
        passed = abs(rank_ic) > 0.02 and abs(rank_ic_ir) > 0.5 and abs(t_stat) > 2

        return {
            "task": task_name,
            "factor": factor_name,
            "days": days,
            "Rank_IC": rank_ic,
            "Rank_IC_IR": rank_ic_ir,
            "t統計量": t_stat,
            "result" : f"Rank_IC={rank_ic:.2f}, Rank_IC_IR={rank_ic_ir:.2f}, t統計量={t_stat:.2f}",
            "passed": passed,
            "error": None,
        }
    except Exception as exc:
        return {
            "task": task_name,
            "factor": None,
            "days": days,
            "Rank_IC": np.nan,
            "Rank_IC_IR": np.nan,
            "t統計量": np.nan,
            "passed": False,
            "error": f"{type(exc).__name__}: {exc}",
        }


In [8]:
# 單因子可行性驗證：不吞例外，把 vol_std（22 日 → 欄位 vol_std_1M）完整跑一遍
import traceback
from IPython.display import display

base_name, days = "vol_std", zj.DAYS_1M  # 22；task 名會顯示成 vol_std_22d
generator = getattr(zj, f"generate_{base_name}")

before = set(df.columns)
factor_df = generator(df, days)
new_columns = [c for c in factor_df.columns if c not in before]
assert len(new_columns) == 1, new_columns
factor_name = new_columns[0]

print(f"因子欄位：{factor_name}")
print(f"非空列數：{factor_df[factor_name].drop_nulls().len():,} / {factor_df.height:,}")
print(f"untradable==0 列數：{factor_df.filter(pl.col('untradable') == 0).height:,}")
display(factor_df.select(["date", "symbol", factor_name, "1day_return", "untradable"]).head())

try:
    alpha = single_factor_eval.run_single_factor_test(
        factor_df,
        factor_name=factor_name,
        group_count=5,
        show_plots=False,
    )
    print("IC 統計：")
    display(alpha.df_info2)
except Exception:
    traceback.print_exc()


因子欄位：vol_std_1M
非空列數：5,172,984 / 5,213,808
untradable==0 列數：4,366,139


date,symbol,vol_std_1M,1day_return,untradable
date,str,f64,f64,i8
2015-06-01,"""1101""",null,-0.006188,0
2015-06-02,"""1101""",null,-0.033624,0
2015-06-03,"""1101""",null,0.003866,0
2015-06-04,"""1101""",null,0.02439,0
2015-06-05,"""1101""",null,-0.026316,0


多空分組統計


,年化收益率,超额年化,最大回撤,超额最大回撤,年化波动,超额年化波动,换手率,月度胜率,超额月度胜率,跟踪误差,夏普比率,信息比率
分组1,11.93%,-2.66%,24.46%,49.53%,8.69%,8.55%,5.11%,71.76%,41.98%,0.0054,1.0851,-0.3120
分组2,13.64%,-0.95%,33.21%,22.77%,12.71%,4.75%,11.39%,67.94%,46.56%,0.0030,0.8768,-0.2024
分组3,15.43%,0.83%,40.10%,8.07%,16.27%,2.46%,12.61%,64.12%,60.31%,0.0015,0.7950,0.3384
分组4,14.85%,0.25%,45.91%,14.51%,19.51%,4.41%,11.44%,65.65%,55.73%,0.0028,0.6331,0.0562
分组5,17.16%,2.55%,51.79%,30.16%,24.15%,10.09%,5.08%,62.60%,52.67%,0.0064,0.6070,0.2533
t統計量,5.22%,5.22%,51.27%,51.27%,18.09%,18.09%,5.09%,55.73%,55.73%,0.0114,0.1505,0.2887
多空组合2,1.21%,1.21%,27.41%,27.41%,8.63%,8.63%,11.41%,52.67%,52.67%,0.0054,-0.1494,0.1401


IC 統計


,vol_std_1M
IC_mean,0.0065
Rank_IC,-0.0462
IC_std,0.1747
IC_IR,0.0371
Rank_IC_IR,-0.2373
IR,0.2756
P(IC<-0.02),54.87%
P(IC>0.02),36.00%
t統計量,1.9136
p-value,0.0558


IC 統計：


,vol_std_1M
IC_mean,0.0065
Rank_IC,-0.0462
IC_std,0.1747
IC_IR,0.0371
Rank_IC_IR,-0.2373
IR,0.2756
P(IC<-0.02),54.87%
P(IC>0.02),36.00%
t統計量,1.9136
p-value,0.0558


In [9]:
from tqdm.auto import tqdm

results = []
for task in tqdm(factor_tasks, desc="批量回測"):
    result = _run_factor(task)
    results.append(result)
    state = "通過" if result["passed"] else ("失敗" if result["error"] else "未通過")
    if result["error"]:
        print('有東西失敗')
        break
    
    tqdm.write(f"{result['task']}: {result['result']}")

print("全部回測完成")


批量回測:   0%|          | 0/95 [00:00<?, ?it/s]

vol_std_22d: Rank_IC=-0.05, Rank_IC_IR=-0.24, t統計量=1.91
vol_std_66d: Rank_IC=-0.05, Rank_IC_IR=-0.23, t統計量=2.03
vol_std_126d: Rank_IC=-0.05, Rank_IC_IR=-0.23, t統計量=1.65
vol_down_std_22d: Rank_IC=-0.03, Rank_IC_IR=-0.19, t統計量=1.80
vol_down_std_66d: Rank_IC=-0.04, Rank_IC_IR=-0.20, t統計量=2.25
vol_down_std_126d: Rank_IC=-0.04, Rank_IC_IR=-0.20, t統計量=1.94
vol_up_std_22d: Rank_IC=-0.05, Rank_IC_IR=-0.31, t統計量=0.93
vol_up_std_66d: Rank_IC=-0.05, Rank_IC_IR=-0.29, t統計量=1.32
vol_up_std_126d: Rank_IC=-0.05, Rank_IC_IR=-0.28, t統計量=1.29
vol_highlow_avg_22d: Rank_IC=-0.05, Rank_IC_IR=-0.23, t統計量=2.21
vol_highlow_avg_66d: Rank_IC=-0.04, Rank_IC_IR=-0.21, t統計量=2.27
vol_highlow_avg_126d: Rank_IC=-0.04, Rank_IC_IR=-0.21, t統計量=1.84
vol_highlow_std_22d: Rank_IC=-0.05, Rank_IC_IR=-0.33, t統計量=1.09
vol_highlow_std_66d: Rank_IC=-0.05, Rank_IC_IR=-0.30, t統計量=1.84
vol_highlow_std_126d: Rank_IC=-0.05, Rank_IC_IR=-0.30, t統計量=1.74
vol_upshadow_avg_22d: Rank_IC=-0.04, Rank_IC_IR=-0.19, t統計量=2.79
vol_upshadow_avg_6

In [16]:
results_df = pd.DataFrame(results).reset_index(drop=True)

failed_results = results_df[results_df["error"].notna()].copy()

# 最終篩選：|Rank_IC| > 0.04 且 |t統計量| > 2
mask = (
    results_df["error"].isna()
    & (results_df["Rank_IC"].abs() > 0.04)
    & (results_df["t統計量"].abs() > 2)
)
selected_results = (
    results_df.loc[mask, ["factor", "days", "Rank_IC", "Rank_IC_IR", "t統計量"]]
    .sort_values("Rank_IC", key=lambda s: s.abs(), ascending=False)
    .reset_index(drop=True)
)

print(f"通過門檻（|Rank_IC|>0.04 且 |t|>2）：{len(selected_results)} / {len(results_df)}")
if len(failed_results):
    print(f"執行失敗：{len(failed_results)}（詳情在 failed_results）")
selected_results


通過門檻（|Rank_IC|>0.04 且 |t|>2）：6 / 95


,factor,days,Rank_IC,Rank_IC_IR,t統計量
0,vol_highlow_avg_1M,22.0,-0.0465,-0.2261,2.2143
1,vol_std_3M,66.0,-0.0461,-0.2281,2.0320
2,vol_highlow_avg_3M,66.0,-0.0443,-0.2109,2.2661
3,vol_upshadow_std_6M,126.0,-0.0432,-0.2581,2.2213
4,vol_upshadow_std_3M,66.0,-0.0424,-0.2568,2.6407
5,vol_upshadow_std_1M,22.0,-0.0413,-0.2638,2.6743


In [11]:
results

[{'task': 'vol_std_22d',
  'factor': 'vol_std_1M',
  'days': 22,
  'Rank_IC': -0.0462,
  'Rank_IC_IR': -0.2373,
  't統計量': 1.9136,
  'result': 'Rank_IC=-0.05, Rank_IC_IR=-0.24, t統計量=1.91',
  'passed': False,
  'error': None},
 {'task': 'vol_std_66d',
  'factor': 'vol_std_3M',
  'days': 66,
  'Rank_IC': -0.0461,
  'Rank_IC_IR': -0.2281,
  't統計量': 2.032,
  'result': 'Rank_IC=-0.05, Rank_IC_IR=-0.23, t統計量=2.03',
  'passed': False,
  'error': None},
 {'task': 'vol_std_126d',
  'factor': 'vol_std_6M',
  'days': 126,
  'Rank_IC': -0.0459,
  'Rank_IC_IR': -0.2271,
  't統計量': 1.6501,
  'result': 'Rank_IC=-0.05, Rank_IC_IR=-0.23, t統計量=1.65',
  'passed': False,
  'error': None},
 {'task': 'vol_down_std_22d',
  'factor': 'vol_down_std_1M',
  'days': 22,
  'Rank_IC': -0.0332,
  'Rank_IC_IR': -0.1925,
  't統計量': 1.8017,
  'result': 'Rank_IC=-0.03, Rank_IC_IR=-0.19, t統計量=1.80',
  'passed': False,
  'error': None},
 {'task': 'vol_down_std_66d',
  'factor': 'vol_down_std_3M',
  'days': 66,
  'Rank_IC': -

In [12]:
failed_results

,task,factor,days,Rank_IC,Rank_IC_IR,t統計量,result,passed,error
